In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

In [2]:
df = pd.read_excel("Test2_REALIGNED_OVERWRITE.xlsx")
df.columns = df.columns.str.strip()

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values(["cusip","date"])

# numeric cleanup
for col in ["spread","price","sduration","coupon","ytm"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# keep needed columns only
df = df.dropna(subset=["date","spread","price","sduration"])

# restrict period
df = df[df["date"].between("2017-01-01","2019-12-31")]
df["ym"] = df["date"].dt.to_period("M")

price_col = "price"

In [3]:
first_day = (
    df.sort_values("date")
      .groupby(["cusip","ym"])
      .first()
      .reset_index()
)

# next month price + next month label
first_day["next_price"] = first_day.groupby("cusip")[price_col].shift(-1)
first_day["next_ym"]    = first_day.groupby("cusip")["ym"].shift(-1)

# drop incomplete rows
first_day = first_day.dropna(subset=["next_price","next_ym"])

# enforce valid month pairs
first_day = first_day[
    first_day["ym"].between("2017-01","2019-12") &
    first_day["next_ym"].between("2017-01","2019-12")
]

In [4]:
# price return
first_day["price_ret"] = (
    (first_day["next_price"] - first_day[price_col]) /
     first_day[price_col]
)

# carry = coupon%/12 / price
first_day["carry"] = ((first_day["coupon"] / 100) / 12) / first_day[price_col]

# total return
first_day["ret"] = first_day["price_ret"] + first_day["carry"]

# DTS
first_day["dts"] = first_day["spread"] * first_day["sduration"]

# wide monthly return matrix
ret_pivot = (
    first_day.pivot(index="ym", columns="cusip", values="ret")
    .sort_index()
)

In [5]:
def optimize_weights(mu, cov, w_max=0.05):
    """Maximize Sharpe ratio with weight caps."""
    n = len(mu)
    w0 = np.ones(n) / n

    def neg_sharpe(w):
        ret = np.dot(w, mu)
        vol = np.sqrt(np.dot(w.T, np.dot(cov, w)))
        if vol <= 0:
            return 1e9
        return -(ret / vol)

    bounds = [(0, w_max)] * n
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]

    res = minimize(
        neg_sharpe, w0,
        method="SLSQP",
        bounds=bounds,
        constraints=cons,
        options={"maxiter": 500, "ftol": 1e-9}
    )

    return res.x

In [6]:
port_low  = []
port_mid  = []
port_high = []

months = sorted(first_day["ym"].unique())

for idx in range(3, len(months)):
    ym = months[idx]

    # trailing 3 months
    window = months[idx-3:idx]
    hist = ret_pivot.loc[window]

    # current month bonds
    month_df = first_day[first_day["ym"] == ym].copy()
    month_df = month_df.sort_values("dts")
    n = len(month_df)
    if n < 6:
        continue

    k = n // 3
    low  = month_df.iloc[:k].copy()
    mid  = month_df.iloc[k:2*k].copy()
    high = month_df.iloc[2*k:3*k].copy()

    # helper
    def prepare_bucket(bucket):
        cus = list(bucket["cusip"])
        sub = hist[cus].dropna(axis=1, how="any")

        if sub.shape[1] < 2:
            return None, None, None

        cusips = list(sub.columns)
        bucket_ordered = bucket.set_index("cusip").loc[cusips].reset_index()

        mu  = sub.mean().values
        cov = np.cov(sub.T)

        return bucket_ordered, mu, cov

    # prepare each
    low_b,  mu_low,  cov_low  = prepare_bucket(low)
    mid_b,  mu_mid,  cov_mid  = prepare_bucket(mid)
    high_b, mu_high, cov_high = prepare_bucket(high)

    if low_b is None or mid_b is None or high_b is None:
        continue

    # optimize
    w_low  = optimize_weights(mu_low,  cov_low)
    w_mid  = optimize_weights(mu_mid,  cov_mid)
    w_high = optimize_weights(mu_high, cov_high)

    # realized return
    ret_low  = np.dot(w_low,  low_b["ret"].values)
    ret_mid  = np.dot(w_mid,  mid_b["ret"].values)
    ret_high = np.dot(w_high, high_b["ret"].values)

    # append
    port_low.append({
        "month": ym,
        "ret": ret_low,
        "cusips": list(low_b["cusip"]),
    })

    port_mid.append({
        "month": ym,
        "ret": ret_mid,
        "cusips": list(mid_b["cusip"]),
    })

    port_high.append({
        "month": ym,
        "ret": ret_high,
        "cusips": list(high_b["cusip"]),
    })

In [7]:
low_opt  = pd.DataFrame(port_low)
mid_opt  = pd.DataFrame(port_mid)
high_opt = pd.DataFrame(port_high)

low_opt["cum_ret"]  = (1 + low_opt["ret"]).cumprod() - 1
mid_opt["cum_ret"]  = (1 + mid_opt["ret"]).cumprod() - 1
high_opt["cum_ret"] = (1 + high_opt["ret"]).cumprod() - 1

print("LOW DTS:")
print(low_opt.tail(), "\n")

print("MID DTS:")
print(mid_opt.tail(), "\n")

print("HIGH DTS:")
print(high_opt.tail())

LOW DTS:
      month       ret                                             cusips  \
27  2019-07  0.009356  [37045XBN5, 254010AC5, 38143CCX7, 06406RAJ6, 2...   
28  2019-08  0.006965  [254010AC5, 37045XBN5, 38143CCX7, 06406RAJ6, 2...   
29  2019-09  0.000520  [254010AC5, 38143CCX7, 06406RAJ6, 25470DAQ2, 2...   
30  2019-10 -0.004590  [38143CCX7, 06406RAJ6, 25470DAQ2, 25468PDK9, 1...   
31  2019-11 -0.001116  [38143CCX7, 06406RAJ6, 25468PDK9, 25470DAQ2, 8...   

     cum_ret  
27  0.027489  
28  0.034645  
29  0.035184  
30  0.030433  
31  0.029282   

MID DTS:
      month       ret                                             cusips  \
27  2019-07  0.016319  [10948WAA1, 68389XAM7, 85172FAN9, 88579YAH, 44...   
28  2019-08  0.019504  [85172FAN9, 68389XAM7, 88579YAH, 444454AF9, 21...   
29  2019-09 -0.006267  [85172FAN9, 444454AF9, 63938CAJ7, 68389XAM7, 8...   
30  2019-10 -0.002223  [444454AF9, 85172FAN9, 100743AJ2, 88579YAH, 21...   
31  2019-11  0.007451  [444454AF9, 68389XAM7, 100743A